In [7]:
import requests
import time
import pandas

API_KEY = ""

url = f"http://ws.audioscrobbler.com/2.0/?method=chart.gettoptracks&api_key={YOUR_API_KEY}&format=json&limit=100&page=1"

all_artists = []

for i in range (1,101):
    params = {
        "method": "chart.gettopartists",
        "api_key": API_KEY,
        "format": "json",
        "limit": 100,
        "page": i
    }
    response = requests.get(url, params=params)
    data = response.json()
    artists = data['artists']['artist']
    all_artists.extend(artist['name'] for artist in artists)
    # time.sleep(.25)

print(all_artists)
        




['PinkPantheress', 'The Weeknd', 'Kendrick Lamar', 'Radiohead', 'Taylor Swift', 'Rihanna', 'Kanye West', 'Charli xcx', 'Lady Gaga', 'Drake', 'Tyler, The Creator', 'Tame Impala', 'Ariana Grande', 'Lana Del Rey', 'Sabrina Carpenter', 'Bad Bunny', 'Bruno Mars', 'Gorillaz', 'Arctic Monkeys', 'Billie Eilish', 'Harry Styles', 'Frank Ocean', 'Paramore', 'Mitski', 'A$AP Rocky', 'Doja Cat', 'Chappell Roan', 'Don Toliver', 'Fleetwood Mac', 'The Smiths', 'Linkin Park', 'Deftones', 'Justin Bieber', 'The Neighbourhood', 'The Marías', 'SZA', 'Travis Scott', 'Coldplay', 'Childish Gambino', 'Michael Jackson', 'Olivia Rodrigo', 'Playboi Carti', 'Clairo', 'TV Girl', 'Future', 'Zara Larsson', 'Twenty One Pilots', 'sombr', 'Beyoncé', 'Mac Miller', 'Nirvana', 'Olivia Dean', 'Britney Spears', 'Joji', 'Lil Uzi Vert', 'Katy Perry', 'The Beatles', 'beabadoobee', 'Lorde', 'Kali Uchis', 'Maroon 5', 'Addison Rae', 'Mac DeMarco', 'The Cure', 'Laufey', 'Tate McRae', 'My Chemical Romance', 'Red Hot Chili Peppers', '

In [ ]:
url = "http://ws.audioscrobbler.com/2.0/"
result = []
for page in range (1,11):
    params = {
        "method": "chart.gettoptracks",
        "api_key": API_KEY,
        "format": "json",
        "limit": 1000,
        "page": page
    }
    response = requests.get(url, params=params)
    data = response.json()
    tracks = data.get("tracks", {}).get("track", [])
    
    for track in tracks:
        result.append({
            "artist": track["artist"]["name"],
            "track": track["name"],
            "playcount": track.get("playcount"),
            "listeners": track.get("listeners")
        })
        time.sleep(.05)

print(result)

In [40]:
df = pandas.DataFrame(result)
df.to_csv("songs.csv", index=False)

In [27]:
df = pandas.DataFrame()
df['artist_name'] = all_artists

df.to_csv("artist_names.csv",index=False)

In [28]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

API_KEY = "bd5cff0c8431c3b8888f8db83ed41ace"
BASE_URL = "http://ws.audioscrobbler.com/2.0/"

# 1. Get top artists (you can paginate)
def get_top_artists(limit=1000, page=1):
    params = {
        "method": "chart.gettopartists",
        "api_key": API_KEY,
        "format": "json",
        "limit": limit,
        "page": page
    }
    response = requests.get(BASE_URL, params=params).json()
    return [artist["name"] for artist in response.get("artists", {}).get("artist", [])]

# 2. Get top tracks for an artist
def get_top_tracks(artist_name, limit=10):
    params = {
        "method": "artist.gettoptracks",
        "artist": artist_name,
        "api_key": API_KEY,
        "format": "json",
        "limit": limit
    }
    response = requests.get(BASE_URL, params=params).json()
    tracks = response.get("toptracks", {}).get("track", [])
    results = []
    for track in tracks:
        track_name = track["name"]
        listeners = track.get("listeners")
        playcount = track.get("playcount")

        # 3. Get top tags for the track
        tags = []
        track_info_params = {
            "method": "track.getInfo",
            "artist": artist_name,
            "track": track_name,
            "api_key": API_KEY,
            "format": "json"
        }
        track_info = requests.get(BASE_URL, params=track_info_params).json()
        if "track" in track_info and "toptags" in track_info["track"]:
            tags = [tag["name"] for tag in track_info["track"]["toptags"].get("tag", [])]

        results.append({
            "artist": artist_name,
            "track": track_name,
            "listeners": listeners,
            "playcount": playcount,
            "tags": ", ".join(tags)
        })
    return results

# 4. Multithreaded scraping
all_artists = []
for page in range(1, 11):  # first 10 pages, adjust as needed
    all_artists.extend(get_top_artists(limit=100, page=page))

song_data = []

with ThreadPoolExecutor(max_workers=20) as executor:  # adjust threads
    future_to_artist = {executor.submit(get_top_tracks, artist): artist for artist in all_artists}
    for future in as_completed(future_to_artist):
        artist = future_to_artist[future]
        try:
            tracks = future.result()
            if tracks:
                song_data.extend(tracks)
                print(f"Finished {artist} with {len(tracks)} tracks")
            else:
                print(f"No tracks for {artist}")
        except Exception as e:
            print(f"Error for {artist}: {e}")

# 5. Save to CSV
df = pd.DataFrame(song_data)
df.to_csv("lastfm_top_tracks_multithreaded.csv", index=False)
print(f"Saved {len(song_data)} tracks")

Finished Rihanna with 10 tracks
Finished Kendrick Lamar with 10 tracks
Finished Gorillaz with 10 tracks
Finished Lady Gaga with 10 tracks
Finished Taylor Swift with 10 tracks
Finished Charli xcx with 10 tracks
Finished Lana Del Rey with 10 tracks
Finished Arctic Monkeys with 10 tracks
Finished Ariana Grande with 10 tracks
Finished PinkPantheress with 10 tracks
Finished Tame Impala with 10 tracks
Finished Tyler, The Creator with 10 tracks
Finished Billie Eilish with 10 tracks
Finished Bad Bunny with 10 tracks
Finished The Weeknd with 10 tracks
Finished Kanye West with 10 tracks
Finished Radiohead with 10 tracks
Finished Drake with 10 tracks
Finished Bruno Mars with 10 tracks
Finished Paramore with 10 tracks
Finished Fleetwood Mac with 10 tracks
Finished A$AP Rocky with 10 tracks
Finished Sabrina Carpenter with 10 tracks
Finished Frank Ocean with 10 tracks
Finished Harry Styles with 10 tracks
Finished Mitski with 10 tracks
Finished The Smiths with 10 tracks
Finished Justin Bieber with 10